In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
import shutil
from PIL import Image
from pprint import pprint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#Change your directory to wherever your folder is
os.chdir("/content/drive/MyDrive/Colab Notebooks/PyTorch/Project")

In [ ]:
cwd = os.getcwd()

# Folder in your Google Drive
folder_name = 'pokemon_sprites'
output_folder = os.path.join(cwd, folder_name)

def sprite_folder(output_folder):
    name_folder = output_folder.split("/")[-1]
    if os.path.exists(output_folder):
        print(f"Folder ({name_folder}) is present! Checking if valid")
        try:
            file_list = os.listdir(output_folder)
            pngs_only = [f for f in file_list if f.endswith(".png")]

            if file_list and (len(file_list) == len(pngs_only)):
                delete_img = input(f"Pokemon Sprites already exist in folder ({len(pngs_only)} pngs present), would you like to clear the folder? 'y' for yes, anything else for no:\n")
                delete_img = delete_img.lower()
                if delete_img == "y":
                    shutil.rmtree(output_folder)
                    os.makedirs(output_folder)
                else:
                    print("Using Previously Loaded Sprites")
            else:
                print(f"'{name_folder}' Folder is either empty or does not contain all PNGs. Clearing folder")
                shutil.rmtree(output_folder)
                os.makedirs(output_folder)
        except Exception as e:
            print(f"Error occured! {e}")
            print("Clearing Folder")
            shutil.rmtree(output_folder)
            os.makedirs(output_folder)
    else:
        print(f"Folder ({name_folder}) doesn't exist! Creating Folder")
        os.makedirs(output_folder)

sprite_folder(output_folder)

data_path = 'pokemon_data.txt'
dex_num_path = 'pokedex_numbers.txt'

pkmn_data_path = os.path.join(cwd, data_path)
pkdx_nums_path = os.path.join(cwd, dex_num_path)

## Typing infrastructure

In [ ]:
type_storage = {
         "grass" : None,
         "fire" : None,
         "water" : None,
         "bug" : None,
         "normal" : None,
         "poison" : None,
         "electric": None,
         "ground" : None,
         "fairy" : None,
         "fighting": None,
         "psychic": None,
         "rock" : None,
         "ghost": None,
         "ice": None,
         "dragon": None,
         "dark": None,
         "steel":None,
         "flying": None
    }

In [ ]:
type_path = os.path.join(cwd, 'Typing_sprites')
for filename in os.listdir(type_path):
    image_path = os.path.join(type_path, filename)
    end_path = (((((image_path.split("\\"))[-1]).split("-"))[1]).split("_")[0])
    selected_type = end_path[:-2:].lower()
    image = Image.open(image_path)
    type_storage[selected_type] = image

## API Call

In [ ]:
def initial_call():
    #API that includes every pokemon (limit 100000 is well above the amount that actually exist)
    url = "https://pokeapi.co/api/v2/pokemon?limit=100000"

    response = requests.get(url)
    if response.status_code == 200:
        data_all = response.json()  # Converts JSON response to a Python dict
        formatted_output = json.dumps(data_all, indent=4)
        pokemon_data = data_all["results"]
        return pokemon_data

In [ ]:
def collect_pokemon_data():
    try:
        with open(pkmn_data_path, 'r') as file:
            content = file.read()
            if not content.strip():
                data_exists = False
                reload_data = False
            else:
                all_pokemon = json.loads(content)
                data_exists = True
                reload_data = False
                print("The Pokemon data already exists and is ready to use!")
                reset = input("If you are sure you want to reload the Pokemon data input 'y', otherwise input any other key")
                reset = reset.lower()
                if reset == "y":
                    reload_data = True
                else:
                    with open(pkdx_nums_path, 'r') as file:
                        pokedex_content = file.read()
                        pokedex_nums = json.loads(pokedex_content)

                    print("Previous content loaded")
                    return all_pokemon, pokedex_nums


    except FileNotFoundError:
        data_exists = False


    if (data_exists == False) or (reload_data == True):
        if data_exists == False:
            cont = input("Currently there is no pokemon data, would you like to pull data from the PokeAPI?\nInput 'y' to proceed, otherwise input any other character:")
            cont = cont.lower()
            if cont != "y":
                print("Did not call the API!")
                return
        print("Calling the API")
        #Dictionaries that data will be inputted into
        all_pokemon = {}
        pokedex_nums = {}
        pokemon_data = initial_call()
        #Goes through every entry in PokeAPI
        for index, data in enumerate(tqdm(pokemon_data)):
            #Setting form to false, pokemon have have different forms
            #Assume it is the original form until later specified
            form = False
            #Pokemon name
            pokemon = data["name"]
            #Link to other API with more in depth information for specific pokemon
            extra_data = data["url"]
            #Send a request to gett data
            response_pkmn = requests.get(extra_data)
            if response_pkmn.status_code == 200:
                #Converting to JSON string
                pkmn_data = response_pkmn.json()
                #Species contains the pokedex number, but must be extracted
                species_info = pkmn_data["species"]
                #Pokedex number (Identifying number for each pokemon (stays the same between forms))
                pokedex_num = (species_info["url"].split("/"))[-2]
                #Extracting name of species, valuable for finding base form of a form/variant
                species = species_info["name"]

                #Pokemon can have either 1 or 2 types
                types = pkmn_data["types"]
                pkmn_types = []
                for i in types:
                    pkmn_type = i["type"]["name"]
                    pkmn_types.append(pkmn_type)


                if pokedex_num not in pokedex_nums:
                    pokedex_nums[pokedex_num] = {"regular": pokemon,
                                                "form": []}
                elif pokedex_num in pokedex_nums:
                    form = True
                    pokedex_nums[pokedex_num]["form"].append(pokemon)

                all_pokemon[pokemon] = {"pokedex": pokedex_num,
                                        "type" : pkmn_types,
                                        "form": form}

                sprite_url = pkmn_data["sprites"]["other"]["official-artwork"]["front_default"]

                if sprite_url:
                    response = requests.get(sprite_url)
                    # Checking if valid response code
                    if response.status_code == 200:
                        # Read the image content from the response
                        img_data = response.content
                        # Save like before
                        if form == False:
                            file_path = os.path.join(output_folder, f"{pokedex_num}.png")
                        elif form == True:
                            #Removing the species(base pokemon name) from the full pokemon name(includes form)
                            form_name = pokemon.replace(species, "").lstrip("-")
                            form_dex_entry = f"{pokedex_num}-{form_name}.png"
                            file_path = os.path.join(output_folder, form_dex_entry)

                        with open(file_path, "wb") as f:
                            f.write(img_data)

                        all_pokemon[pokemon]["image_file"] = file_path
        formatted_pkmn_data = json.dumps(all_pokemon, indent = 4)
        with open(pkmn_data_path, 'w') as file:
            file.write(formatted_pkmn_data)

        formatted_pokedex_nums = json.dumps(pokedex_nums, indent = 4)
        with open(pkdx_nums_path, 'w') as file:
            file.write(formatted_pokedex_nums)

        print("Data Extracted!")
        return all_pokemon, pokedex_nums

In [ ]:
all_pokemon, pokedex_nums = collect_pokemon_data()

In [ ]:
print(len(all_pokemon))

In [ ]:
#Checking frequency of each type
from collections import defaultdict

def simple_type_count(data):
    type_count = defaultdict(int)
    for mon in data.values():
        for typ in mon["type"]:
            type_count[typ] += 1
    return dict(type_count)

In [ ]:
#Checking frequency of each type
from collections import defaultdict

def simple_type_count_df(data):
    type_count = defaultdict(int)
    for mon in data.values():
        for typ in mon["type"]:
            type_count[typ] += 1

    dict_return = dict(type_count)
    return dict_return, pd.DataFrame(dict_return.items(), columns = ['Type', 'Count']).sort_values(by='Count', ascending = False)

In [ ]:
type_counts, type_counts_df = simple_type_count_df(all_pokemon)
#Checking prevalence of each type
pprint(type_counts)

In [ ]:
def class_distribution_plot(class_balance_df, set_name):
    plt.figure(figsize=(12, 6))
    bars = plt.bar(class_balance_df['Type'], class_balance_df['Count'], color='skyblue')
    plt.xlabel('Pokémon Type')
    plt.ylabel('Count')
    plt.title(f'{set_name} Class Balance by Pokémon Type')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.grid(axis='y', linestyle='--', alpha=0.5)

    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,  # x-coordinate (center of bar)
            height,                             # y-coordinate (just above bar)
            f'{int(height)}',                   # label text
            ha='center', va='bottom', fontsize=10
        )

    plt.show()

class_distribution_plot(type_counts_df, "All Pokemon Data (Original)")

## Handling forms

In [ ]:
generation_split = {
    1: [1, 151],
    2: [152, 251],
    3: [252, 386],
    4: [387, 493],
    5: [494, 649],
    6: [650, 721],
    7: [722, 809],
    8: [810, 905],
    9: [906, 1025]
}

In [ ]:
#Checking if any pokemon/pokemon forms don't have a sprite
#Will exclude these forms
missing_image_file = [name for name, data in all_pokemon.items() if "image_file" not in data]
for name in missing_image_file:
    print(f"{name} → Type: {all_pokemon[name]['type']}, Pokedex: {all_pokemon[name]['pokedex']}")

In [ ]:
form_mons = []
for i in pokedex_nums:
    entry = pokedex_nums[i]
    forms = entry["form"]
    if len(forms) == 0:
        continue
    else:
        pokemon = entry["regular"]
        output = [pokemon, i]
        for form in forms:
            form_mons.append(output)
            break

In [ ]:
import copy
def remove_bad_forms(pokedex):
    #Making a deep copy to make a new version of pokedex with only the forms wanted
    cleaned_pokedex = copy.deepcopy(pokedex)
    accept_forms = ["mega", "alola", "hisui", "galar", "paldea", "primal", "origin"]
    reject_forms = ["gmax", "totem", "build", "mode", "cap", "busted"]
    for entry in form_mons:
        pkmn = entry[0]
        pkdx = entry[1]
        last_mon = ""
        for form in pokedex[pkdx]["form"]:
            if (form.split("-"))[-1] in reject_forms or (form.split("-")[-2] in reject_forms):
                cleaned_pokedex[pkdx]["form"].remove(form)
            elif (form.split("-"))[-1] in accept_forms or (form.split("-"))[-2] == "mega":
                continue
            #Not removing any generation 9 pokemon as all will be used in test data
            elif int(pkdx) >= generation_split[9][0]:
                continue
            else:
                if pkmn != last_mon:
                    print(f"Pokemon: {pkmn}")
                    last_mon = pkmn
                print(f"\t{form}")
                remove = input("Remove? y for yes, any other key for no:\n")
                remove = remove.lower()
                if remove == "y":
                    cleaned_pokedex[pkdx]["form"].remove(form)
                else:
                    continue
    return cleaned_pokedex

In [ ]:
#Will be using remove_bad_forms function to remove certain forms
#There are many purely cosmetic forms, however unless they are water types (most prevalent class)
#I will attempt to keep them, unless they are extremely unnecessary

cleaned_pkdx_path = os.path.join(cwd, 'cleaned_pokedex.txt')

try:
    with open(cleaned_pkdx_path, 'r') as file:
        content = file.read()
        if not content.strip():
            print("File is empty, must filter through forms!\n")
            cleaned_pokedex = remove_bad_forms(pokedex_nums)
            formatted_clean_pkdx = json.dumps(cleaned_pokedex, indent=4)
            with open(cleaned_pkdx_path, 'w') as file:
                file.write(formatted_clean_pkdx)
            print("Data Saved!")
        else:
            cont = input("A cleaned pokedex with certain forms removed already exists!\nIf you want to create a new version input 'y', otherwise input any other key to load previous data:\n")
            cont = cont.lower()
            if cont == "y":
                cleaned_pokedex = remove_bad_forms(pokedex_nums)
                formatted_clean_pkdx = json.dumps(cleaned_pokedex, indent=4)
                with open(cleaned_pkdx_path, 'w') as file:
                    file.write(formatted_clean_pkdx)
                print("Data Saved!")
            else:
                cleaned_pokedex = json.loads(content)
                print("Data Loaded!")
except FileNotFoundError:
    print("File does not exist! Beginning form filtering process")
    cleaned_pokedex = remove_bad_forms(pokedex_nums)
    formatted_clean_pkdx = json.dumps(cleaned_pokedex, indent=4)
    with open(cleaned_pkdx_path, 'w') as file:
        file.write(formatted_clean_pkdx)
    print("Data Saved!")


### Type Multi-Hot Encoding

In [ ]:
def typing_encoding(typing):
    #Each type will have a specific index
    type_index = {
         "grass" : 0,
         "fire" : 1,
         "water" : 2,
         "bug" : 3,
         "normal" : 4,
         "poison" : 5,
         "electric": 6,
         "ground" : 7,
         "fairy" : 8,
         "fighting": 9,
         "psychic": 10,
         "rock" : 11,
         "ghost": 12,
         "ice": 13,
         "dragon": 14,
         "dark": 15,
         "steel": 16,
         "flying": 17
    }

    #Making an array of 18 zeros
    #Represents the typing
    output = np.zeros(18)
    #Changing certain indeces to 1 to represent a pokemons typing
    for t in typing:
        curr_type = t.lower()
        index = type_index[curr_type]
        output[index] = 1
    return output.tolist()


In [ ]:
train_data = {}
test_data = {}
train = False
for i in cleaned_pokedex:
    index = int(i)
    entry = cleaned_pokedex[i]
    base = entry["regular"]
    if generation_split[9][0] > index:
        #TRAIN (Not gen 9) Normal
        if base not in train_data:
            train_data[base] = {"pkdx" : index}
        train_data[base]["type"] = all_pokemon[base]["type"]
        train_data[base]["image_path"] = all_pokemon[base]["image_file"]
        train = True
    elif generation_split[9][0] <= index:
        #TEST (gen 9) Normal
        if base not in test_data:
            test_data[base] = {"pkdx" : index}
        test_data[base]["type"] = all_pokemon[base]["type"]
        test_data[base]["image_path"] = all_pokemon[base]["image_file"]
        train = False
    forms = entry["form"]
    if len(forms) > 0:
        for form in forms:
            segments = form.split("-")
            #Some generation 9 pokemon are new forms of all pokemon, called Paldean forms
            #Based on their pokedex they are not considered generation 9, but they were introduced in the newest generation
            if train == True and ("paldea" in segments):
                #TEST (not gen 9) paldea form
                if form not in test_data:
                    test_data[form] = {"pkdx" : index}
                test_data[form]["type"] = all_pokemon[form]["type"]
                test_data[form]["image_path"] = all_pokemon[form]["image_file"]
            elif train == False:
                #TEST (gen 9) form
                if form not in test_data:
                    test_data[form] = {"pkdx" : index}
                test_data[form]["type"] = all_pokemon[form]["type"]
                test_data[form]["image_path"] = all_pokemon[form]["image_file"]
            elif train == True:
                #TRAIN (not gen 9) form
                if form not in train_data:
                    train_data[form] = {"pkdx" : index}
                train_data[form]["type"] = all_pokemon[form]["type"]
                train_data[form]["image_path"] = all_pokemon[form]["image_file"]

In [ ]:
#Adding multi-hot encoder for labeling our types
data_sets = [train_data, test_data]

for data in data_sets:
    for pkmn in data:
        data[pkmn]['label'] = typing_encoding(data[pkmn]['type'])


test_data_path = os.path.join(cwd, 'test_data_pretensor.txt')

test_data_json = json.dumps(test_data, indent=4)
with open(test_data_path, 'w') as file:
    file.write(test_data_json)
print("Test Data Saved!")

In [ ]:
print(f"Amount of samples in training data: {len(train_data)}")
print(f"Amount of samples in test data: {len(test_data)}")

## Oversampling

In [ ]:
def type_distribution(data):
    type_frequency = {
            "grass" : {'amount' : 0, 'to_generate' : None},
            "fire" : {'amount' : 0, 'to_generate' : None},
            "water" : {'amount' : 0, 'to_generate' : None},
            "bug" : {'amount' : 0, 'to_generate' : None},
            "normal" : {'amount' : 0, 'to_generate' : None},
            "poison" : {'amount' : 0, 'to_generate' : None},
            "electric": {'amount' : 0, 'to_generate' : None},
            "ground" : {'amount' : 0, 'to_generate' : None},
            "fairy" : {'amount' : 0, 'to_generate' : None},
            "fighting": {'amount' : 0, 'to_generate' : None},
            "psychic": {'amount' : 0, 'to_generate' : None},
            "rock" : {'amount' : 0, 'to_generate' : None},
            "ghost": {'amount' : 0, 'to_generate' : None},
            "ice": {'amount' : 0, 'to_generate' : None},
            "dragon": {'amount' : 0, 'to_generate' : None},
            "dark": {'amount' : 0, 'to_generate' : None},
            "steel": {'amount' : 0, 'to_generate' : None},
            "flying": {'amount' : 0, 'to_generate' : None}
        }

    type_pokemon =  {
            "grass" : {"original":[], "to_augment" : []},
            "fire" : {"original":[], "to_augment" : []},
            "water" : {"original":[], "to_augment" : []},
            "bug" : {"original":[], "to_augment" : []},
            "normal" : {"original":[], "to_augment" : []},
            "poison" : {"original":[], "to_augment" : []},
            "electric": {"original":[], "to_augment" : []},
            "ground" : {"original":[], "to_augment" : []},
            "fairy" : {"original":[], "to_augment" : []},
            "fighting": {"original":[], "to_augment" : []},
            "psychic": {"original":[], "to_augment" : []},
            "rock" : {"original":[], "to_augment" : []},
            "ghost": {"original":[], "to_augment" : []},
            "ice": {"original":[], "to_augment" : []},
            "dragon": {"original":[], "to_augment" : []},
            "dark": {"original":[], "to_augment" : []},
            "steel": {"original":[], "to_augment" : []},
            "flying": {"original":[], "to_augment" : []}
        }
    for pkmn in data:
        typing = data[pkmn]['type'][:]
        for i, typ in enumerate(typing):
            type_frequency[typ]['amount'] += 1
            if len(typing) == 1:
                entry = (pkmn, [])
            else:
                if i == 0:
                    entry = (pkmn, [typing[1]])
                elif i == 1:
                    entry = (pkmn, [typing[0]])
            type_pokemon[typ]['original'].append(entry)

    max_type = max(type_frequency, key = lambda x: type_frequency[x]['amount'])
    max_amount = type_frequency[max_type]['amount']

    for typ in type_frequency:
        amount_present = type_frequency[typ]['amount']
        to_generate = max_amount - amount_present
        type_frequency[typ]['to_generate'] = to_generate

    return type_frequency, type_pokemon

In [ ]:
from pprint import pprint
type_frequency, pokemon_type_augment = type_distribution(train_data)

pprint(type_frequency)

In [ ]:
def type_generate_distribution(data):
    # Sort types by amount descending
    sorted_types = sorted(data.items(), key=lambda x: x[1]['amount'], reverse=True)

    types = [t[0] for t in sorted_types]
    amount = [t[1]['amount'] for t in sorted_types]
    to_generate = [t[1]['to_generate'] for t in sorted_types]
    max_values = [a + g for a, g in zip(amount, to_generate)]

    x = np.arange(len(types))

    plt.figure(figsize=(14,7))

    bars1 = plt.bar(x, amount, color='blue', label='Amount')
    bars2 = plt.bar(x, to_generate, bottom=amount, color='red', label='To Generate')

    # Amount values inside blue bars
    for i, val in enumerate(amount):
        plt.text(x[i], val - val*0.05, str(val), ha='center', va='top', color='white', fontweight='bold')

    # To_generate values inside red bars
    for i, (a, g) in enumerate(zip(amount, to_generate)):
        if g > 5:  # only display if bar is tall enough to fit text nicely
            plt.text(x[i], a + g - g*0.15, str(g), ha='center', va='top', color='white', fontweight='bold')

    # Max values on top of stacked bars
    for i, val in enumerate(max_values):
        plt.text(x[i], val + 3, str(val), ha='center', color='black', fontweight='bold')

    plt.xticks(x, types, rotation=45, ha='right')
    plt.ylabel('Count')
    plt.title('Amount and To Generate by Pokémon Type (Sorted by Amount)')
    plt.legend()

    plt.tight_layout(pad=3)
    plt.subplots_adjust(top=0.85)
    plt.ylim(0, max(max_values) + 10)

    plt.show()


type_generate_distribution(type_frequency)

In [ ]:
import random
import copy
def pokemon_to_augment(type_frequency_original, augment_dict_original):

    type_frequency = copy.deepcopy(type_frequency_original)
    augment_dict = copy.deepcopy(augment_dict_original)
    max_duplicates = 2
    #Keeping track on which pokemon is selected for augmentation and how many times
    pkmn_selection_counter = {}
    #Storing a list of all pokemon that will be augmented
    full_augment_list = []
    #Setting round count as this will be used for randomization
    round = 1

    total_entries = []

    stopped_early = False
    steel_count = 0
    #Checking to see if any types still need to have samples generated
    while any(value['to_generate'] > 0 for value in type_frequency.values()):
        #Making sure a pokemon can only be used once per round
        #Since there's randomness and pokemon with multiple types this is important
        used_this_round = set()
        added_this_round = False

        for typ in type_frequency:
            amount_needed = type_frequency_original[typ]['to_generate']
            gen_amount = type_frequency[typ]['to_generate']
            #If it has reached it threshold of 156 samples, it doesn't need more
            if gen_amount == 0:
                continue
            #randomizing the order so I don't just sample the first entries
            randomized_originals = augment_dict[typ]['original'][:]
            #Making it reproducible by setting the seed to be the round count
            random.seed(f"{typ}-{round}")
            random.shuffle(randomized_originals)

            #iterating through a randomized order of each pokemon for our current type
            for pkmn, secondary_lst in randomized_originals:
                pkmn_count = total_entries.count(pkmn)
                if pkmn_count == max_duplicates:
                    continue
                typing = [typ] + secondary_lst

                if pkmn in used_this_round:
                    continue

                #If statement to handle accepted monotypes
                if len(typing) == 1:
                    if type_frequency[typ]['to_generate'] > 0:
                        augment_dict[typ]['to_augment'].append(pkmn)
                        type_frequency[typ]['to_generate'] -= 1
                        type_frequency[typ]['amount'] += 1
                        total_entries.append(pkmn)
                        used_this_round.add(pkmn)
                        added_this_round = True
                    continue

                #Getting secondary type out of secondary list
                secondary = secondary_lst[0]

                #If the secondary type has met its quota it will skip
                if type_frequency[secondary]['to_generate'] == 0:
                    continue

                #If statement to handle accepted dual types
                elif len(typing) == 2:
                    if (type_frequency[typ]['to_generate'] == 0) or (type_frequency[secondary]['to_generate'] == 0):
                        continue
                    augment_dict[typ]['to_augment'].append(pkmn)
                    augment_dict[secondary]['to_augment'].append(pkmn)
                    type_frequency[typ]['to_generate'] -= 1
                    type_frequency[secondary]['to_generate'] -= 1
                    type_frequency[typ]['amount'] += 1
                    type_frequency[secondary]['amount'] += 1
                    total_entries.append(pkmn)
                    used_this_round.add(pkmn)
                    added_this_round = True

        if not added_this_round:
            print(f"Stopped early at round {round}: no further augmentation possible without exceeding max duplicates.")
            stopped_early = True
            break


        round += 1

    return augment_dict, type_frequency, total_entries

In [ ]:
final_augment_dict, new_type_frequency, to_augment_list = pokemon_to_augment(type_frequency, pokemon_type_augment)

In [ ]:
print(f"Amount of images augmented to augment: {len(to_augment_list)}")

In [ ]:
pprint(new_type_frequency)

In [ ]:
type_generate_distribution(new_type_frequency)

## Creating Augmentations

In [ ]:
from collections import Counter
from torchvision import transforms
from PIL import Image
import os
from tqdm import tqdm
import torchvision.transforms.functional as TF
import shutil

#Transformations that are applied to augment

# transforms.RandomHorizontalFlip()
# transforms.RandomRotation(45)
# transforms.RandomAffine(degrees=0, translate=(0.1,0.1), scale=(0.8,1.2), shear=10)
# transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.3, hue=0.1)

train_data_edit = copy.deepcopy(train_data)

create_augment = False

#Output directory
output_dir = os.path.join(cwd, 'augmented_sprites')

train_data_path = os.path.join(cwd, 'train_data_pretensor.txt')

#Checking if augmented_sprites folder exists
if os.path.exists(output_dir):
    #If it does it counts the amount of images
    png_count = sum(1 for file in os.listdir(output_dir) if file.endswith(".png"))
else:
    #If it doesn't exist, creates folder and sets create_augment to True
    print("Augmented sprite folder does not exist. Creating all augmented sprites now.")
    os.makedirs(output_dir, exist_ok=True)
    create_augment = True


expected_length = len(train_data) + len(to_augment_list)
train_data_final = {}
#Checking if train_data_final.txt exists
if os.path.exists(train_data_path):
    #Open file
    with open(train_data_path, 'r') as file:
        #Load data if possible
        try:
            train_data_final = json.load(file)
            if len(train_data_final) != expected_length:
                print(f"Train data does not have the correct amount of entries. Creating augmentations!")
                create_augment = True
        #If error loading the flle
        except json.JSONDecodeError:
            print("train_data_final.txt exists but couldn't be parsed as JSON. Recreating augmentations!")
            loaded_train_data = {}
            #Create_augment set to True
            create_augment = True
#train_data_final.txt does not exist
else:
    print("Train data with augmentations is not stored. Creating augmentations!")
    #Create_augment set to True
    create_augment = True

if not create_augment:
    if png_count == len(to_augment_list):
        response = input(f"Augmented sprites already exist ({len(to_augment_list)}) PNGs found).\nWould you like to recreate them? Input 'y' for yes, anything else for no:\n")
        if response.strip().lower() != 'y':
            print("Skipping augmentation.")
        else:
            shutil.rmtree(output_dir)  #deletes all existing files
            os.makedirs(output_dir, exist_ok=True)
            create_augment = True
    else:
        print(f"Only {png_count} PNGs found. Not all augmented sprites exist, creating all augmented sprites now.")
        shutil.rmtree(output_dir)
        os.makedirs(output_dir, exist_ok=True)
        create_augment = True



if create_augment:
    #Load your list and count duplicates
    pkmn_counts = Counter(to_augment_list)
    #Augment and save
    for pkmn, count in tqdm(pkmn_counts.items()):
        sprite_path = all_pokemon.get(pkmn, {}).get("image_file", None)

        if sprite_path is None or not os.path.exists(sprite_path):
            print(f"Missing sprite for {pkmn}")
            continue

        #Getting the label and type list for the augmented pokemon
        original_label = train_data[pkmn]['label']
        original_type = train_data[pkmn]['type']
        pkdx = train_data[pkmn]['pkdx']
        try:
            image_rgba = Image.open(sprite_path).convert("RGBA")
            r, g, b, a = image_rgba.split()
            rgb_image = Image.merge("RGB", (r, g, b))

            for i in range(count):
                #Generate consistent random params
                angle = random.uniform(-45, 45)
                translate = (random.uniform(-0.1, 0.1) * rgb_image.size[0],
                            random.uniform(-0.1, 0.1) * rgb_image.size[1])
                scale = random.uniform(0.8, 1.2)
                shear = random.uniform(-10, 10)

                #Apply transforms to RGB and Alpha identically
                rgb_aug = TF.affine(rgb_image, angle=angle, translate=translate, scale=scale, shear=shear)
                a_aug = TF.affine(a, angle=angle, translate=translate, scale=scale, shear=shear)

                #Color jitter only on RGB
                rgb_aug = transforms.ColorJitter(
                    brightness=0.5, contrast=0.5, saturation=0.3, hue=0.1
                )(rgb_aug)

                #Half the time image will be flipped horizontally
                if random.random() < 0.5:
                    rgb_aug = TF.hflip(rgb_aug)
                    a_aug = TF.hflip(a_aug)

                #Merge back
                augmented_image = Image.merge("RGBA", (*rgb_aug.split(), a_aug))


                #filename for sprite of non-augmented pokemon
                filename = os.path.basename(sprite_path)
                # Remove the file extension
                sprite_id = os.path.splitext(filename)[0]
                suffix = f"augmented_{i}" if i > 0 else "augmented"
                augmented_pkdx = f"{sprite_id}-{suffix}"
                augmented_pkmn = f"{pkmn}-{suffix}"


                augmented_filename = f"{augmented_pkdx}.png"
                save_path = os.path.join(output_dir, augmented_filename)
                augmented_image.save(save_path)

                train_data_edit[augmented_pkmn] = {'image_path': save_path,
                                                   'label': original_label,
                                                   'type': original_type,
                                                   'pkdx': pkdx}
        except Exception as e:
            print(f"Error augmenting {pkmn}: {e}")

    train_data_json = json.dumps(train_data_edit, indent=4)
    with open(train_data_path, 'w') as file:
        file.write(train_data_json)
    print("\nData Saved!")

## Neural Network
### (If data is loaded code can be run from here, as a result some functions are reinitialized)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, f1_score
import seaborn as sns
from torch.utils.data import random_split
import os
import json
import copy
from tqdm import tqdm
from pprint import pprint
from PIL import Image
from torch.utils.data import random_split
from google.colab import drive

In [ ]:
drive.mount('/content/drive')
os.chdir("/content/drive/MyDrive/Colab Notebooks/PyTorch/Project")

In [ ]:
#Checking frequency of each type
from collections import defaultdict

def simple_type_count_df(data):
    type_count = defaultdict(int)
    for mon in data.values():
        for typ in mon["type"]:
            type_count[typ] += 1

    dict_return = dict(type_count)
    return dict_return, pd.DataFrame(dict_return.items(), columns = ['Type', 'Count']).sort_values(by='Count', ascending = False)

In [ ]:
cwd = os.getcwd()

train_data_path = os.path.join(cwd, 'train_data_pretensor.txt')
test_data_path = os.path.join(cwd, 'test_data_pretensor.txt')

def convert_labels_to_tensors(data_dict):
    for name, data in data_dict.items():
        if isinstance(data['label'], list):
            data['label'] = torch.tensor(data['label'], dtype=torch.float32)

try:
    with open(train_data_path, 'r') as file:
        train_data_pretensor = json.load(file)
        convert_labels_to_tensors(train_data_pretensor)
        print("Train Data Loaded!")
except Exception as e:
    print("Train data is not stored. Error:", e)

try:
    with open(test_data_path, 'r') as file:
        test_data_pretensor = json.load(file)
        convert_labels_to_tensors(test_data_pretensor)
        print("Test Data Loaded!")
except Exception as e:
    print("Test data is not stored. Error:", e)

train_length = len(train_data_pretensor)

test_length = len(test_data_pretensor)

In [ ]:
#Length of train data
print(f"Amount of samples in Training Data (Pre-Validation Split): {train_length}")
print(f"Amount of samples in Test Data: {test_length}")
#Class presence
train_class_balance, train_class_df = simple_type_count_df(train_data_pretensor)

In [ ]:
def class_distribution_plot(class_balance_df, set_name):
    plt.figure(figsize=(12, 6))
    bars = plt.bar(class_balance_df['Type'], class_balance_df['Count'], color='skyblue')
    plt.xlabel('Pokémon Type')
    plt.ylabel('Count')
    plt.title(f'{set_name} Class Balance by Pokémon Type')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.grid(axis='y', linestyle='--', alpha=0.5)

    for bar in bars:
        height = bar.get_height()
        if height >= 20:
            offset = height * 0.05
        else:
            offset = 0
        plt.text(
            bar.get_x() + bar.get_width() / 2,  # x-coordinate (center of bar)
            height - offset,            # y-coordinate (slightly below bar if above 20, otherwise above)
            f'{int(height)}',                   # label text
            ha='center', va='bottom', fontsize=10
        )

    plt.show()

class_distribution_plot(train_class_df, "Train")

In [ ]:
test_class_balance, test_class_df = simple_type_count_df(test_data_pretensor)

class_distribution_plot(test_class_df, "Test")

In [ ]:
#convert data to torch.FloatTensor

#Define your transform
transform = transforms.Compose([
    # Resize directly to 224 x 224 as that's the expected input for resnet
    transforms.Resize((224, 224)),
    # Convert PIL Image to tensor with shape [C, H, W]
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
    ])


train_final_path = os.path.join(cwd, 'train_data_final.pt')
test_final_path = os.path.join(cwd, 'test_data_final.pt')

generate_tensors = False

try:
    train_data_final = torch.load(train_final_path)
    if len(train_data_final) == train_length:
        print("Train data loaded")
    else:
        print("Train file present but missing entries!")
        generate_tensors = True
except Exception as e:
    print("Final train data is not stored. Error:", e)
    generate_tensors = True

if not generate_tensors:
    try:
        test_data_final = torch.load(test_final_path)
        if len(test_data_final) == test_length:
            print("Test data loaded")
        else:
            print("Test file present but missing entries!")
            generate_tensors = True
    except Exception as e:
        print("Final test data is not stored. Error:", e)
        generate_tensors = True

if not generate_tensors:
    proceed = input("Data already seems to exist!\nBut if you want to regenerate tensors input'y', if not enter anything else:\n")
    proceed = proceed.lower()
    if proceed == 'y':
        generate_tensors = True


if generate_tensors:
    proceed = input("Data does not exist!\nWould you like to generate tensors? input'y', if not enter anything else:\n")
    proceed = proceed.lower()
    if proceed == 'y':
        #Making tensors of all selected sprites for training
        train_data_final = copy.deepcopy(train_data_pretensor)
        for name, info in tqdm(train_data_final.items()):
            img = Image.open(info["image_path"]).convert("RGB")
            tensor_img = transform(img)
            train_data_final[name]["tensor"] = tensor_img
        try:
            torch.save(train_data_final, train_final_path)
            print("Train data stored")
        except Exception as e:
            print("Error occured saving training data", e)
        print("Train Data ready for training!")

        #Test data
        test_data_final = copy.deepcopy(test_data_pretensor)

        for name, info in tqdm(test_data_final.items()):
            img = Image.open(info["image_path"]).convert("RGB")
            tensor_img = transform(img)
            test_data_final[name]["tensor"] = tensor_img

        try:
            torch.save(test_data_final, test_final_path)
            print("Train data stored")
        except Exception as e:
            print("Error occured saving testing data", e)
        print("Test data ready for model evaluation!")


## Initializing models and Data Loader

In [ ]:
class PokemonDataset(Dataset):
    def __init__(self, data):
        self.samples = list(data.items())

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        name, entry = self.samples[idx]
        image_tensor = entry["tensor"]
        label_tensor = entry["label"]

        if "image_path" in entry:
            return image_tensor, label_tensor, name, entry["image_path"]
        else:
            return image_tensor, label_tensor, name

torch.manual_seed(42)

# Initialize Dataset and DataLoader
full_train_dataset = PokemonDataset(train_data_final)

#Taking 10% of training data for validation
val_size = int(0.1 * len(full_train_dataset))
#Removing validation size from training data to get new train size
train_size = len(full_train_dataset) - val_size
#Randomly splitting the data for the respective sizes
train_subset, val_subset = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_subset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=8, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Criterion/Loss function will be the same for all models
criterion = nn.BCEWithLogitsLoss()
#Storage for trained models
os.makedirs("models", exist_ok=True)

### Custom CNN

In [ ]:
#CNN class
class PokemonCNN(nn.Module):
    def __init__(self, num_classes=18):
        super(PokemonCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),  # (B, 32, H, W)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # (B, 32, H/2, W/2)

            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # (B, 64, H/2, W/2)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # (B, 64, H/4, W/4)

            nn.Conv2d(64, 128, kernel_size=3, padding=1),  # (B, 128, H/4, W/4)
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # (B, 128, H/8, W/8)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),  # (B, 128 * H/8 * W/8)
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)  # Output raw logits
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_cnn = PokemonCNN()

model_cnn = PokemonCNN().to(device)
#defining optimizer for CNN
optimizer_cnn = torch.optim.Adam(model_cnn.parameters(), lr=1e-3, weight_decay=1e-4)

### Resnet 18

In [ ]:
#Resnet 18 Model Original
from torchvision.models import resnet18, ResNet18_Weights
#Setting device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Load pre-trained model & freeze the layers
model_res = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
for param in model_res.parameters():
    param.requires_grad = False

model_res.fc = nn.Linear(model_res.fc.in_features, 18)


# Move the model to the GPU if available
model_res =  model_res.to(device)
#Ensuring model is in train mode
model_res.train()

#Defining optimizer for ResNet 18

#Only updating unfrozen layers
optimizer_res = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_res.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)


### Resnet 50

In [ ]:
#Resnet 50 Model Original
from torchvision.models import resnet50, ResNet50_Weights
#Setting device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Load pre-trained model & freeze the layers
model_res50 = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
for param in model_res50.parameters():
    param.requires_grad = False

model_res50.fc = nn.Linear(model_res50.fc.in_features, 18)


# Move the model to the GPU if available
model_res50 =  model_res50.to(device)
#Ensuring model is in train mode
model_res50.train()


#Defining optimizer for ResNet 50

#Only updating unfrozen layers
optimizer_res50 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_res50.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)


### Type handling (Image storage and Label to Type Conversion)
#### Repeating certain previous sections, as Neural Network section of code can run independetly if files are alreayd present

In [ ]:
type_storage = {
         "grass" : None,
         "fire" : None,
         "water" : None,
         "bug" : None,
         "normal" : None,
         "poison" : None,
         "electric": None,
         "ground" : None,
         "fairy" : None,
         "fighting": None,
         "psychic": None,
         "rock" : None,
         "ghost": None,
         "ice": None,
         "dragon": None,
         "dark": None,
         "steel":None,
         "flying": None
    }

type_path = os.path.join(cwd, 'Typing_sprites')

for filename in os.listdir(type_path):
    image_path = os.path.join(type_path, filename)
    end_path = (((((image_path.split("\\"))[-1]).split("-"))[1]).split("_")[0])
    selected_type = end_path[:-2:].lower()
    image = Image.open(image_path)
    type_storage[selected_type] = image

In [ ]:
type_to_index = {
         "grass" : 0,
         "fire" : 1,
         "water" : 2,
         "bug" : 3,
         "normal" : 4,
         "poison" : 5,
         "electric": 6,
         "ground" : 7,
         "fairy" : 8,
         "fighting": 9,
         "psychic": 10,
         "rock" : 11,
         "ghost": 12,
         "ice": 13,
         "dragon": 14,
         "dark": 15,
         "steel": 16,
         "flying": 17
    }

index_to_type = {
    0: "grass",
    1: "fire",
    2: "water",
    3: "bug",
    4: "normal",
    5: "poison",
    6: "electric",
    7: "ground",
    8: "fairy",
    9: "fighting",
    10: "psychic",
    11: "rock",
    12: "ghost",
    13: "ice",
    14: "dragon",
    15: "dark",
    16: "steel",
    17: "flying"
}


In [ ]:
def typing_encoding(typing):
    #Making an array of 18 zeros
    #Represents the typing
    output = np.zeros(18)
    #Changing certain indeces to 1 to represent a pokemons typing
    for t in typing:
        curr_type = t.lower()
        index = type_to_index[curr_type]
        output[index] = 1
    return output.tolist()


def tensor_to_type(label, img_display = False):
    indices = (label == 1).nonzero(as_tuple=True)[0]
    indices_list = indices.tolist()

    typing = []
    type_sprites = []
    for ind in indices_list:
        typ = (index_to_type[ind])
        type_sprite = type_storage[typ]
        if img_display:
            display(type_sprite)
        typing.append(typ)
        type_sprites.append(type_sprite)

    return typing



### Examining Validation Split

In [ ]:
#Stratify not an option with the random_split option in PyTorch
#Ensuring the validation set is somewhat balanced, and remaining training data is as well
def validation_split_values(model, val_loader, display_img = True):
    validation_set = {}
    #setting resnet model to eval mode
    model.eval()
    with torch.no_grad():
        for images, labels, names, image_paths in tqdm(val_loader):
            images = images.to(device)
            outputs = model(images)


            for i in range(len(names)):
                pkmn = names[i]
                label = labels[i]
                typing = tensor_to_type(label)
                file_path = image_paths[i]
                validation_set[pkmn] = {
                    'image_path' : file_path,
                    'label' : label,
                    'type' : typing
                }
                if display_img:
                    img = Image.open(file_path)

                    print(f"\n{pkmn.capitalize()}")

                    # Show image
                    display(img)

    return(validation_set)

validation_data = validation_split_values(model_res, val_loader)

In [ ]:
train_data_final_dict = {
    name: data
    for name, data in train_data_pretensor.items()
    if name not in validation_data
}

print(f"Amount of samples in Final Training Set: {len(train_data_final_dict)}")

In [ ]:
train_final_classes, train_final_class_df = simple_type_count_df(train_data_final_dict)

class_distribution_plot(train_final_class_df, "Final Train")

In [ ]:
validation_classes, validation_class_df = simple_type_count_df(validation_data)

class_distribution_plot(validation_class_df, "Validation")

### Handling predicted probabilities and evalutiation

In [ ]:
#Getting top 4 type predictions and their probabilities
def typing_probability(probs):
    all_preds = []
    dict_list = []
    for i, entry in enumerate(probs):
        #getting top n (4) predictions
        k = 4
        #finding top k values and indices for probability vector
        topk_vals, topk_indices = torch.topk(entry, k=k)
        #Getting top k preds, which will be displayed for each output
        topk_preds = torch.zeros_like(entry)
        topk_preds[topk_indices] = 1

        #Creating dictionary for storage
        type_probs = {}

        #iterationg through all type indices for the topk predictions
        for ind,entry in enumerate(topk_indices):
            type_index = entry.item()
            #using type_index to get pokemon type
            typ = index_to_type[type_index]
            count = ind + 1
            probability = topk_vals[ind]
            type_probs[count] = {"type": typ,
                                 "index": type_index,
                                 "prob": probability,
                                 }

        type_probs["topk_tensor"] = topk_preds
        dict_list.append(type_probs)


    return dict_list


In [ ]:
def safe_deepcopy_dict(d):
    result = {}
    for k, v in d.items():
        if isinstance(v, torch.Tensor):
            result[k] = v.detach().clone()  # detach and clone to avoid graph issues
        elif isinstance(v, dict):
            result[k] = safe_deepcopy_dict(v)  # recursive copy for nested dicts
        else:
            result[k] = v
    return result

In [ ]:
#Function for evaluating predicted label vs actual label
#2 types of accuracy we're testing
#Exact match and Top K match
def pred_label_eval(pred_dict, labels):
    total_points_topk = 0.0
    correct_preds_topk = 0.0

    total_points_exact = 0.0
    correct_preds_exact = 0.0

    #Store updated dicts per sample for reuse
    updated_exact_dicts = []
    updated_topk_dicts = []
    for i, label in enumerate(labels):
        current_prediction = pred_dict[i]

        #Checking to see if monotype or dual type
        num_types = torch.sum(label).item()
        if num_types == 1:
            #monotypes are given 3 predictions for the true label
            allowed_preds = 3
        else:
            #dualtypes are given 4 predictions for the true label
            allowed_preds = 4

        #making a deep copy to not alter top k original topk prediction dictionary
        pred_updated_topk = safe_deepcopy_dict(current_prediction)
        pred_updated_exact = safe_deepcopy_dict(current_prediction)

        del pred_updated_exact["topk_tensor"]
        del pred_updated_topk["topk_tensor"]


        #Exact handling (pred selection)
        if num_types == 1:
            pred_updated_exact.pop(2, None)
            pred_updated_exact.pop(3, None)
            pred_updated_exact.pop(4, None)
        elif num_types == 2:
            pred_updated_exact.pop(3, None)
            pred_updated_exact.pop(4, None)


        #Topk handling (pred selection)

        #if monotype delete 4th prediction
        if allowed_preds == 3:
            #delete 4th prediction
            pred_updated_topk.pop(4, None)

        for i in range(allowed_preds):
            num = i+1
            #Always include 1 prediction bare minimum
            if num == 1:
                continue
            #if a dual type always include 2 predictions bare minimum
            if num == 2 and (allowed_preds == 4):
                continue
            #finding probability of prediction (applicable to preds 2,3 for monotype and 3,4 for dual types)
            #If not present assumed to be 0
            probability = pred_updated_topk.get(num, {'prob': 0})['prob']
            #If that probability is below 10%, will be disregarded even as a top 2/3/4 prediction
            if probability < 0.1:
                pred_updated_topk.pop(num, None)



        #Exact handling
        valid_preds_exact = torch.zeros_like(label)

        for key, entry in pred_updated_exact.items():
            if key == "topk_tensor":
                continue
            typing = entry['type']
            index = type_to_index[typing]
            valid_preds_exact[index] = 1


        #Topk handling
        valid_preds_topk = torch.zeros_like(label)

        for key, entry in pred_updated_topk.items():
            if key == "topk_tensor":
                continue
            typing = entry['type']
            index = type_to_index[typing]
            valid_preds_topk[index] = 1

        #Adding prediction labels to dict
        pred_updated_exact["pred_label"] = valid_preds_exact
        pred_updated_topk["pred_label"] = valid_preds_topk

        #Save updated dicts for this sample
        updated_exact_dicts.append(pred_updated_exact)
        updated_topk_dicts.append(pred_updated_topk)



        #Evaluating true label vs predicted label
        type_indices = torch.nonzero(label, as_tuple=True)[0]

        #Checking correctness of prediction
        correct_pred_exact = 0
        correct_pred_topk = 0

        #Checking indicies of non-zero values of label
        for indx in type_indices:
            #If present in valid_preds, adds 1 to correct_prediction score
            if valid_preds_exact[indx] == 1:
                correct_pred_exact += 1
            if valid_preds_topk[indx] == 1:
                correct_pred_topk += 1

        #To prevent monotypes from getting half the points for a correct prediction, multiplying correct_pred_topk value by 2
        if num_types == 1:
            correct_pred_topk *= 2
            correct_pred_exact *= 2

        #Not correct: Score = 0
        #Partial correct (dual types only): Score = 1
        #Fully correct: Score = 2

        correct_preds_exact += correct_pred_exact
        correct_preds_topk += correct_pred_topk

        total_points_exact += 2
        total_points_topk += 2

    ratio_exact = (correct_preds_exact, total_points_exact)
    ratio_topk = (correct_preds_topk, total_points_topk)

    return ratio_exact, ratio_topk, updated_exact_dicts, updated_topk_dicts

### Validation Set

In [ ]:
def validation(model, val_loader, criterion, device):
    model.eval()  # Set model to evaluation mode
    val_loss = 0.0

    correct_exact = 0.0
    total_exact = 0.0

    correct_topk = 0.0
    total_topk = 0.0

    with torch.no_grad():
        for val_inputs, val_labels, *_ in val_loader:
            val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)

            outputs = model(val_inputs)
            loss = criterion(outputs, val_labels)
            val_loss += loss.item()

            probs = torch.sigmoid(outputs)
            pred_dict = typing_probability(probs)

            ratio_exact, ratio_topk, *_ = pred_label_eval(pred_dict, val_labels)

            correct_exact += ratio_exact[0]
            total_exact += ratio_exact[1]

            correct_topk += ratio_topk[0]
            total_topk += ratio_topk[1]

    avg_loss = val_loss / len(val_loader)
    avg_accuracy_topk = 100 * (correct_topk / total_topk) if total_topk != 0 else 0.0
    avg_accuracy_exact = 100 * (correct_exact / total_exact) if total_exact != 0 else 0.0

    return avg_loss, avg_accuracy_topk, avg_accuracy_exact

### Training model

In [ ]:
def train_model(model,
                optimizer,
                train_loader=train_loader,
                val_loader=val_loader,
                criterion=criterion,
                device=device,
                epochs=100,
                batch_size=16,
                num_logs=5,
                early_stopping=True,
                patience=5,
                min_epochs=20,
                min_delta=0.001):

    train_losses = []
    val_losses = []
    train_accuracies_topk = []
    train_accuracies_exact = []
    val_accuracies_topk = []
    val_accuracies_exact = []

    best_val_loss = float('inf')
    trigger_times = 0

    # Fixed log boundaries
    log_boundaries = [20, 40, 60, 80, 100]

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        accuracy_total_topk = 0
        accuracy_total_exact = 0

        for i, data in enumerate(train_loader):
            inputs, labels, *rest = data
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            probs = torch.sigmoid(outputs)
            pred_dict = typing_probability(probs)

            ratio_exact, ratio_topk, *_ = pred_label_eval(pred_dict, labels)

            batch_accuracy_topk = 100 * (ratio_topk[0] / ratio_topk[1]) if ratio_topk[1] != 0 else 0.0
            batch_accuracy_exact = 100 * (ratio_exact[0] / ratio_exact[1]) if ratio_exact[1] != 0 else 0.0

            accuracy_total_topk += batch_accuracy_topk
            accuracy_total_exact += batch_accuracy_exact

            if i in log_boundaries:
                val_loss, val_accuracy_topk, val_accuracy_exact = validation(model, val_loader, criterion, device)
                print(f"\n[Epoch {epoch+1}, Batch {i+1}]")
                print(f"Train Loss: {loss.item():.4f} | Train Top-K Acc: {batch_accuracy_topk:.2f}% | Train Exact Acc: {batch_accuracy_exact:.2f}%")
                print(f"Val Loss: {val_loss:.4f} | Val Top-K Acc: {val_accuracy_topk:.2f}% | Val Exact Acc: {val_accuracy_exact:.2f}%")

        num_batches = len(train_loader)
        epoch_loss = running_loss / num_batches
        train_accuracy_topk = accuracy_total_topk / num_batches
        train_accuracy_exact = accuracy_total_exact / num_batches

        val_loss, val_accuracy_topk, val_accuracy_exact = validation(model, val_loader, criterion, device)

        train_losses.append(epoch_loss)
        val_losses.append(val_loss)
        train_accuracies_topk.append(train_accuracy_topk)
        train_accuracies_exact.append(train_accuracy_exact)
        val_accuracies_topk.append(val_accuracy_topk)
        val_accuracies_exact.append(val_accuracy_exact)

        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train — Loss: {epoch_loss:.4f} | Top-k Acc: {train_accuracy_topk:.2f}% | Exact Acc: {train_accuracy_exact:.2f}%")
        print(f"Val   — Loss: {val_loss:.4f} | Top-k Acc: {val_accuracy_topk:.2f}% | Exact Acc: {val_accuracy_exact:.2f}%")

        if early_stopping and epoch + 1 >= min_epochs:
            if val_loss + min_delta < best_val_loss:
                best_val_loss = val_loss
                trigger_times = 0
            else:
                trigger_times += 1
                print(f"Validation loss did not improve for {trigger_times} epoch(s).")
                if trigger_times >= patience:
                    print(f"\nEarly stopping triggered at epoch {epoch+1} (no improvement for {patience} epochs).")
                    break

    return (train_losses, train_accuracies_topk, train_accuracies_exact,
            val_losses, val_accuracies_topk, val_accuracies_exact)


#### Resnet 18 model training


In [ ]:
(
    train_losses_res, train_accuracy_topk_res, train_accuracy_exact_res,
    val_losses_res, val_accuracy_topk_res, val_accuracy_exact_res
) = train_model(model_res, optimizer_res)


model_res_storage = [train_losses_res, train_accuracy_topk_res, train_accuracy_exact_res,
                     val_losses_res, val_accuracy_topk_res, val_accuracy_exact_res]

In [ ]:
save_path_res = os.path.join(cwd, 'models/model_resnet18.pth')

torch.save(model_res.state_dict(), save_path_res)
print("Model Saved")

In [ ]:
def graphing_results(model_storage, model_name):
    train_losses = model_storage[0]
    train_accuracy_topk = model_storage[1]
    train_accuracy_exact = model_storage[2]
    val_losses = model_storage[3]
    val_accuracy_topk = model_storage[4]
    val_accuracy_exact = model_storage[5]

    # Plot Losses
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Training Loss', color='orange')
    plt.plot(val_losses, label='Validation Loss', color='blue')
    plt.title(f'Loss Over Epochs {model_name}')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Plot Accuracies Topk
    plt.plot(train_accuracy_exact, label='Train Top-k Accuracy', color='orange')
    plt.plot(val_accuracy_exact, label='Train Exact Accuracy', color='blue')
    plt.title(f'Accuracy Over Epochs (Exact) {model_name}')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    plt.show()

    #Plot Accuracies Exact
    plt.figure(figsize=(10, 5))
    plt.plot(train_accuracy_topk, label='Train Top-k Accuracy', color='orange')
    plt.plot(val_accuracy_topk, label='Val Exact Accuracy', color='blue')
    plt.title(f'Accuracy Over Epochs (Top K) {model_name}')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
graphing_results(model_res_storage, "ResNet 18")

#### Resnet 50 model

In [ ]:
(
    train_losses_res50, train_accuracy_topk_res50, train_accuracy_exact_res50,
    val_losses_res50, val_accuracy_topk_res50, val_accuracy_exact_res50
) = train_model(model_res50, optimizer_res50)


model_res50_storage = [train_losses_res50, train_accuracy_topk_res50, train_accuracy_exact_res50,
                     val_losses_res50, val_accuracy_topk_res50, val_accuracy_exact_res50]

In [ ]:
save_path_res50 = os.path.join(cwd,'models/model_resnet50.pth')

torch.save(model_res50.state_dict(), save_path_res50)
print("Model Saved")

In [ ]:
graphing_results(model_res50_storage, "ResNet 50")

### Custom CNN model

In [ ]:
(
    train_losses_cnn, train_accuracy_topk_cnn, train_accuracy_exact_cnn,
    val_losses_cnn, val_accuracy_topk_cnn, val_accuracy_exact_cnn
) = train_model(model_cnn, optimizer_cnn)


model_cnn_storage = [train_losses_cnn, train_accuracy_topk_cnn, train_accuracy_exact_cnn,
                     val_losses_cnn, val_accuracy_topk_cnn, val_accuracy_exact_cnn]

In [ ]:
graphing_results(model_cnn_storage, "Custom CNN")

In [ ]:
save_path_cnn = os.path.join(cwd,'models/model_cnn.pth')
torch.save(model_cnn.state_dict(), save_path_cnn)
print("Model Saved")

### Evaluating Test Set

In [ ]:
test_dataset = PokemonDataset(test_data_final)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

### Reload models (If you have previously trained models and want to load the, use this section)

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, resnet50, ResNet18_Weights, ResNet50_Weights
import os

def load_model(path, model_type, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    #Architecture selection - MUST MATCH TRAINING
    if model_type == 1:  # ResNet18
        model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)  #Pretrained like training
        for param in model.parameters():
            param.requires_grad = False  #Freeze like training
        model.fc = nn.Linear(model.fc.in_features, 18)  #Only FC layer trainable

    elif model_type == 2:  # ResNet50
        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)  #Pretrained
        for param in model.parameters():
            param.requires_grad = False  # Freeze base layers
        model.fc = nn.Linear(model.fc.in_features, 18)

    elif model_type == 3:  # PokemonCNN
        class PokemonCNN(nn.Module):
            def __init__(self, num_classes=18):
                super(PokemonCNN, self).__init__()

                self.features = nn.Sequential(
                    nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
                    nn.BatchNorm2d(32),
                    nn.ReLU(),
                    nn.MaxPool2d(kernel_size=2),

                    nn.Conv2d(32, 64, kernel_size=3, padding=1),
                    nn.BatchNorm2d(64),
                    nn.ReLU(),
                    nn.MaxPool2d(kernel_size=2),

                    nn.Conv2d(64, 128, kernel_size=3, padding=1),
                    nn.BatchNorm2d(128),
                    nn.ReLU(),
                    nn.MaxPool2d(kernel_size=2),
                )

                self.classifier = nn.Sequential(
                    nn.Flatten(),
                    nn.Linear(128 * 28 * 28, 256),
                    nn.ReLU(),
                    nn.Dropout(0.5),
                    nn.Linear(256, num_classes)
                )

            def forward(self, x):
                x = self.features(x)
                x = self.classifier(x)
                return x

        model = PokemonCNN()
    else:
        raise ValueError("Invalid model_type. Use 1=ResNet18, 2=ResNet50, 3=CNN.")

    #Load weights
    state_dict = torch.load(path, map_location=device)
    model.load_state_dict(state_dict)

    model.to(device)
    model.eval()
    return model




In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.chdir("/content/drive/MyDrive/Colab Notebooks/PyTorch/Project")
cwd = os.getcwd()

load_models = input("Load previous model weights? (y/n): ").lower()
#Load models
if load_models == 'y':
    try:
        #ResNet18
        res18_path = os.path.join(cwd, 'models/model_resnet18.pth')
        if os.path.exists(res18_path):
            model_res = load_model(res18_path, model_type=1)
            print("ResNet18 loaded")
        else:
            print("ResNet18 model file not found")

        #ResNet50
        res50_path = os.path.join(cwd, 'models/model_resnet50.pth')
        if os.path.exists(res50_path):
            model_res50 = load_model(res50_path, model_type=2)
            print("ResNet50 loaded")
        else:
            print("ResNet50 model file not found")

        #PokemonCNN
        cnn_path = os.path.join(cwd, 'models/model_cnn.pth')
        if os.path.exists(cnn_path):
            model_cnn = load_model(cnn_path, model_type=3)
            print("PokemonCNN loaded")
        else:
            print("CNN model file not found")

        print("\nModel loading complete!")

    except Exception as e:
        print(f"Error loading models: {e}")
        print("Check that your model files exist and match the training architecture")

else:
    print("No models loaded")

## Model evaluation

In [ ]:
from sklearn.metrics import multilabel_confusion_matrix, classification_report
import numpy as np

# Generate type_names directly from your index_to_type mapping to ensure correct order
type_names = [index_to_type[i] for i in range(18)]

def test_evaluation_with_metrics(model):
    model.eval()

    all_true = []
    all_pred_exact = []
    all_pred_topk = []

    def convert_eval_dict_to_binary(eval_dict, num_classes=18):
        """Convert evaluation dictionary to binary prediction vector"""
        pred_vector = np.zeros(num_classes, dtype=int)

        # If eval_dict is empty, return all zeros
        if not eval_dict:
            return pred_vector

        for pred_info in eval_dict.values():
            if isinstance(pred_info, dict) and 'type' in pred_info:
                type_name = pred_info['type']
                if type_name in type_names:
                    type_idx = type_names.index(type_name)
                    pred_vector[type_idx] = 1

        return pred_vector

    with torch.no_grad():
        for images, labels, names, image_paths in test_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            pred_dicts = typing_probability(probs)  # Use the same function as your working code

            # Get filtered dicts for exact and topk using your existing function
            ratio_exact, ratio_topk, updated_exact_dicts, updated_topk_dicts = pred_label_eval(pred_dicts, labels)

            for i in range(len(labels)):
                true_label = labels[i].cpu().numpy()

                # Convert your evaluation dictionaries to binary vectors
                exact_dict = updated_exact_dicts[i]
                topk_dict = updated_topk_dicts[i]

                pred_label_exact = convert_eval_dict_to_binary(exact_dict)
                pred_label_topk = convert_eval_dict_to_binary(topk_dict)

                all_true.append(true_label)
                all_pred_exact.append(pred_label_exact)
                all_pred_topk.append(pred_label_topk)

    all_true = np.array(all_true)
    all_pred_exact = np.array(all_pred_exact)
    all_pred_topk = np.array(all_pred_topk)

    # Calculate confusion matrices for exact predictions
    conf_matrices_exact = multilabel_confusion_matrix(all_true, all_pred_exact)

    # Calculate confusion matrices for top-k predictions
    conf_matrices_topk = multilabel_confusion_matrix(all_true, all_pred_topk)

    print("EXACT PREDICTIONS - Classification Report:")
    print("=" * 60)
    print(classification_report(all_true, all_pred_exact, target_names=type_names, zero_division=0))

    print("\nTOP-K PREDICTIONS - Classification Report:")
    print("=" * 60)
    print(classification_report(all_true, all_pred_topk, target_names=type_names, zero_division=0))

    return conf_matrices_exact, conf_matrices_topk

In [ ]:
print("RESNET18")
res_conf_matrix_exact, res_conf_matrix_topk = test_evaluation_with_metrics(model_res)

In [ ]:
print("RESNET50")
res50_conf_matrix_exact, res50_conf_matrix_topk = test_evaluation_with_metrics(model_res50)

In [ ]:
print("CNN")
cnn_conf_matrix_exact, cnn_conf_matrix_topk = test_evaluation_with_metrics(model_cnn)

## Visualizing Results

In [ ]:
def probs_img_output(pred_dict, exact_dict, topk_dict, true_types):
    type_num = len(true_types)
    if type_num == 1:
        mon_type = "MONOTYPE"
        preds_str = "PREDICTION"
    else:
        mon_type = "DUAL-TYPE"
        preds_str = "PREDICTIONS"

    eval_methods = [exact_dict, topk_dict]
    for i, method in enumerate(eval_methods):
        if i == 0:
            title = "EXACT"
        else:
            title = "TOP K"

        print(f"\n___{title} {mon_type} {preds_str}___")
        for key,value in method.items():
            if isinstance(key, int):
                typ = value['type']
                if typ in true_types:
                    flag = "🟩"
                else:
                    flag = "🟥"
                type_sprite = type_storage[typ]
                #print(f"Prediction {key}:")
                display(type_sprite)
                prob = value['prob'].item()
                percentage = prob * 100
                print(f"Probability: {percentage:.2f}% Confidence {flag}")
                last_key = key

        if i != 0:
            #If monotype, and already had 3 predictions can end
            if type_num == 1 and key == 3:
                return
            #If dualtype, and already had 4 preductions can end
            if type_num == 2 and key == 4:
                return
            cont = True
            invalid_k = last_key + 1
            if (type_num == 1 and invalid_k == 4) or invalid_k > 4:
                return
            print(f"\nINVALID PREDICTIONS (Not top {type_num} prediction and below 10% confidence)")

            if type_num == 1:
                max  = 3
            if type_num == 2:
                max = 4

            while cont:
                if invalid_k <= max:
                    if invalid_k in pred_dict:
                        extra_pred = pred_dict[invalid_k]
                        typ = extra_pred['type']
                        type_sprite = type_storage[typ]
                        prob = extra_pred['prob'].item()
                        percentage = prob * 100
                        if typ in true_types:
                            flag = "🟨"
                        else:
                            flag = "🟥"
                        #print(f"Prediction {invalid_k}:")
                        display(type_sprite)
                        print(f"Probability: {percentage:.2f}% Confidence {flag}")
                        invalid_k += 1
                    else:
                        return
                else:
                    return

In [ ]:
def test_evaluation(model):
    model.eval()

    with torch.no_grad():
        correct_exact = 0.0
        total_exact = 0.0
        correct_topk = 0.0
        total_topk = 0.0
        for images, labels, names, image_paths in test_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            pred_dicts = typing_probability(probs)  #returns list of top 4 prediction dictionaries per batch based on probability

            #Get filtered dicts for exact and topk from your updated pred_label_eval
            ratio_exact, ratio_topk, updated_exact_dicts, updated_topk_dicts = pred_label_eval(pred_dicts, labels)

            correct_exact += ratio_exact[0]
            total_exact += ratio_exact[1]

            correct_topk += ratio_topk[0]
            total_topk += ratio_topk[1]

            for i in range(len(names)):
                pkmn = names[i]
                file_path = image_paths[i]
                img = Image.open(file_path)
                label = labels[i]
                count = torch.sum(label).item()

                pred_dict = pred_dicts[i]

                topk_tensor = pred_dict["topk_tensor"]

                #Using evaluation dicts
                exact_dict = updated_exact_dicts[i]
                topk_dict = updated_topk_dicts[i]

                if count == 1:
                    true_type_str = "True Type"
                    pred_type_str_exact = "Exact Predicted Type"
                else:
                    true_type_str = "True Types"
                    pred_type_str_exact = "Exact Predicted Types"

                print(f"\n--- {pkmn.capitalize()} ---")
                display(img)

                print(f"✅ {true_type_str}:")
                true_types = tensor_to_type(label.cpu(), True)

                probs_img_output(pred_dict, exact_dict, topk_dict, true_types)

        exact_accuracy = (correct_exact / total_exact) * 100
        topk_accuracy = (correct_topk / total_topk) * 100

        return exact_accuracy, topk_accuracy

In [ ]:
res_exact_acc, res_topk_acc = test_evaluation(model_res)

In [ ]:
res50_exact_acc, res50_topk_acc = test_evaluation(model_res50)

In [ ]:
cnn_exact_acc, cnn_topk_acc =test_evaluation(model_cnn)

In [ ]:
print(f"Resnet18: Exact Accuracy: {(res_exact_acc):.2f}% | Top K Accuracy: {(res_topk_acc):.2f}%")
print(f"Resnet50: Exact Accuracy: {(res50_exact_acc):.2f}% | Top K Accuracy: {(res50_topk_acc):.2f}%")
print(f"CNN: Exact Accuracy: {(cnn_exact_acc):.2f}% | Top K Accuracy: {(cnn_topk_acc):.2f}%")